# Module 4: George Eliot NLP Pipeline

**Student:** Clay Harris  
**Course:** DS 5001

This notebook creates an F3-level digital analytical edition from three George Eliot novels.

## Set Up

### Configs

In [1]:
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

### Imports

In [2]:
import pandas as pd
import numpy as np
import re
import nltk

### Download NLTK Resources

In [3]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /Users/queclay/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/queclay/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/queclay/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/queclay/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/queclay/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Define Chapter Patterns

Patterns for the three George Eliot novels based on Project Gutenberg formatting.

In [4]:
roman = '[IVXLCM]+'

chap_pats = {
    145: {  # Middlemarch
        'start_line': 127,
        'end_line': 33282,
        'chapter': re.compile(r'^CHAPTER\s+{}\.$'.format(roman))
    },
    6688: {  # The Mill on the Floss
        'start_line': 128,
        'end_line': 21270,
        'chapter': re.compile(r'^Chapter\s+{}\.$'.format(roman))
    },
    507: {  # Adam Bede
        'start_line': 110,
        'end_line': 20700,
        'chapter': re.compile(r'^Chapter\s+{}$'.format(roman))
    }
}

## Import and Chunk Texts

Function to acquire and process the text files into LIBRARY and DOC tables.

In [5]:
def acquire_texts(file_list, chap_pats, OHCO=OHCO):
    """
    Import and chunk text files into library and document tables.
    
    Parameters:
    - file_list: list of file paths
    - chap_pats: dictionary of chapter patterns
    - OHCO: list defining the hierarchy
    
    Returns:
    - library: DataFrame with book metadata
    - docs: DataFrame with paragraphs
    """
    my_lib = []
    my_doc = []

    for file_path in file_list:
        # Extract book ID from filename
        book_id = int(file_path.split('/')[-1].split('-')[0])
        print(f"Processing book_id: {book_id}")
        
        # Read file
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            lines = f.readlines()
        
        df = pd.DataFrame(lines, columns=['line_str'])
        df.index.name = 'line_num'
        df['line_str'] = df['line_str'].str.strip()
        df['book_id'] = book_id
        
        # Normalize punctuation for better tokenization
        df['line_str'] = df['line_str'].str.replace('—', ' — ')
        df['line_str'] = df['line_str'].str.replace('-', ' - ')
        
        # Extract book title - search for it in the first 20 lines
        book_title = None
        for i in range(min(20, len(df))):
            line = df.loc[i, 'line_str']
            # Try standard Gutenberg eBook header
            m = re.search(r'(?:The Project Gutenberg eBook(?: of|,)\s*|Project Gutenberg.s\s*)(.+?)(?:,\s*by.*)?$', line, flags=re.IGNORECASE)
            if m:
                book_title = m.group(1).strip()
                break
        
        # If no Gutenberg header found, look for title after the START marker
        if book_title is None:
            for i in range(min(20, len(df))):
                line = df.loc[i, 'line_str']
                if line and not line.startswith('***') and not line == '' and not re.match(r'^[\d\s]+$', line):
                    book_title = line.strip()
                    break
        
        if book_title is None:
            book_title = f"Book {book_id}"
        
        # Trim to content area
        start = chap_pats[book_id]['start_line']
        end = chap_pats[book_id]['end_line']
        df = df.iloc[start:end]
        
        # Identify chapter boundaries
        chap_pattern = chap_pats[book_id]['chapter']
        chap_lines = df['line_str'].str.match(chap_pattern)
        chap_nums = list(range(1, chap_lines.sum() + 1))
        df.loc[chap_lines, 'chap_num'] = chap_nums
        df['chap_num'] = df['chap_num'].ffill()
        
        # Remove rows without chapter assignment and chapter headings
        df = df[~df['chap_num'].isna()]
        df = df[~chap_lines]
        df['chap_num'] = df['chap_num'].astype('int')
        
        # Group lines by chapter
        df = df.groupby(OHCO[1:2])['line_str'].apply(lambda x: '\n'.join(x)).to_frame()
        
        # Split into paragraphs
        df = df['line_str'].str.split(r'\n\n+', expand=True).stack().to_frame().rename(columns={0: 'para_str'})
        df.index.names = OHCO[1:3]
        df['para_str'] = df['para_str'].str.replace(r'\n', ' ').str.strip()
        df = df[~df['para_str'].str.match(r'^\s*$')]
        
        # Add book_id and reset index
        df['book_id'] = book_id
        df = df.reset_index().set_index(OHCO[:3])
        
        # Register
        my_lib.append((book_id, book_title, file_path))
        my_doc.append(df)
    
    docs = pd.concat(my_doc)
    library = pd.DataFrame(my_lib, columns=['book_id', 'book_title', 'book_file']).set_index('book_id')
    
    return library, docs

### Create LIBRARY and DOC Tables

In [6]:
file_list = [
    '145-0.txt',   # Middlemarch
    '6688-0.txt',  # The Mill on the Floss
    '507-0.txt'    # Adam Bede
]

LIBRARY, DOC = acquire_texts(file_list, chap_pats)

/var/folders/fn/36dz4z514cd0cmgsz1j3_f340000gn/T/ipykernel_38258/225733935.py:70: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~chap_lines]


Processing book_id: 145
Processing book_id: 6688
Processing book_id: 507


In [7]:
# Display LIBRARY table
LIBRARY

,book_title,book_file
book_id,,
145,Middlemarch,145-0.txt
6688,The Mill on the Floss,6688-0.txt
507,Adam Bede,507-0.txt


In [8]:
# Display sample of DOC table
DOC.sample(10)

para_str
book_id chap_num para_num                                                   
145     22       27        “I am at your service, sir, in the matter,” sa...
        39       23        Dorothea felt wretched. She thought her husban...
        47       18        Mr. Casaubon did not preach that morning, and ...
6688    33       41        “I have never had any doubt that you would be ...
        43       13        Mrs Bob’s small nose seemed to be following th...
        37       62        “I give you my word not to meet him or write t...
        21       22        “Oh, Tom!” said Maggie, in a tone of sad remon...
        3        20        “Oh, I’ll tell you what that means. It’s a dre...
145     56       123       “It won’t do to begin making a fuss about one,...
        86       67        But this opinion of his did not cause a lastin...

## Tokenize and Annotate

Convert paragraphs to sentences to tokens with POS tags using NLTK.

In [9]:
def tokenize(doc_df, OHCO=OHCO):
    """
    Tokenize paragraphs into sentences and words with POS tags.
    
    Parameters:
    - doc_df: DataFrame with paragraphs
    - OHCO: list defining the hierarchy
    
    Returns:
    - DataFrame with tokens and POS tags
    """
    # Paragraphs to sentences
    df = doc_df['para_str'].apply(
        lambda x: pd.Series(nltk.sent_tokenize(x))
    ).stack().to_frame().rename(columns={0: 'sent_str'})
    
    # Sentences to tokens with POS tags
    df = df['sent_str'].apply(
        lambda x: pd.Series(nltk.pos_tag(nltk.word_tokenize(x)))
    ).stack().to_frame().rename(columns={0: 'pos_tuple'})
    
    # Extract POS and token from tuple
    df['pos'] = df['pos_tuple'].apply(lambda x: x[1])
    df['token_str'] = df['pos_tuple'].apply(lambda x: x[0])
    
    # Set index
    df.index.names = OHCO
    
    return df

### Create TOKEN Table

In [10]:
TOKEN = tokenize(DOC)
TOKEN.head()

pos_tuple  pos token_str
book_id chap_num para_num sent_num token_num                            
145     1        0        0        0          (Since, IN)   IN     Since
                                   1             (I, PRP)  PRP         I
                                   2            (can, MD)   MD       can
                                   3             (do, VB)   VB        do
                                   4             (no, DT)   DT        no

In [11]:
# Display TOKEN statistics
print(f"Total tokens: {len(TOKEN):,}")
TOKEN.sample(10)

Total tokens: 905,316


pos_tuple  pos token_str
book_id chap_num para_num sent_num token_num                               
145     54       16       6        1             (alone, RB)   RB     alone
507     15       25       1        64             (you, PRP)  PRP       you
6688    12       4        19       41                 (,, ,)    ,         ,
        52       11       0        6             (still, RB)   RB     still
507     15       4        0        36            (large, JJ)   JJ     large
6688    32       1        3        34             (mind, NN)   NN      mind
507     35       3        1        9              (aunt, NN)   NN      aunt
6688    12       3        11       59              (the, DT)   DT       the
145     67       5        2        36         (anything, NN)   NN  anything
        60       28       4        8          (heating, VBG)  VBG   heating

## Create Vocabulary

Extract unique terms and create normalized term strings.

In [12]:
# Create normalized term strings
TOKEN['term_str'] = TOKEN['token_str'].str.lower().str.replace(r'[\W_]+', '', regex=True)

# Create VOCAB from TOKEN
VOCAB = TOKEN['term_str'].value_counts().to_frame().rename(columns={'term_str': 'n'})
VOCAB = VOCAB.sort_index().reset_index().rename(columns={'index': 'term_str'})
VOCAB.index.name = 'term_id'

VOCAB.head()

,term_str,count
term_id,,
0,,145853
1,1,1
2,1790,1
3,1799,2
4,1801,1


In [13]:
# Display VOCAB statistics
print(f"Vocabulary size: {len(VOCAB):,}")
VOCAB.sample(10)

Vocabulary size: 21,486


,term_str,count
term_id,,
18353,supped,1
4413,cumber,5
5719,dreadfully,12
17431,sores,1
21123,winner,1
7617,fowl,10
17710,squyer,1
12973,outstretched,2
18696,tearful,6


## Annotate VOCAB

Add stopwords and Porter stems to the vocabulary.

### Add Stopwords

In [14]:
# Import stopwords from NLTK
sw = pd.DataFrame(nltk.corpus.stopwords.words('english'), columns=['term_str'])
sw = sw.reset_index().set_index('term_str')
sw.columns = ['dummy']
sw['dummy'] = 1

# Map to VOCAB
VOCAB['stop'] = VOCAB['term_str'].map(sw['dummy']).fillna(0).astype('int')

print(f"Stopwords in vocabulary: {VOCAB['stop'].sum()}")

Stopwords in vocabulary: 152


### Add Porter Stems

In [15]:
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()
VOCAB['porter_stem'] = VOCAB['term_str'].apply(stemmer.stem)

VOCAB.sample(10)

,term_str,count,stop,porter_stem
term_id,,,,
3840,consoled,2,0,consol
9455,impregnated,1,0,impregn
10015,intimacy,14,0,intimaci
9597,increasing,5,0,increas
4903,deprecation,3,0,deprec
11231,magsie,14,0,magsi
6760,extraordinary,1,0,extraordinari
5771,drooping,1,0,droop
18148,sua,2,0,sua


### Add pos_max Feature

Find the most frequently associated POS tag for each term.

In [16]:
# Create term_str in TOKEN if not already present
if 'term_str' not in TOKEN.columns:
    TOKEN['term_str'] = TOKEN['token_str'].str.lower().str.replace(r'[\W_]+', '', regex=True)

# Count POS tags for each term
pos_counts = TOKEN.groupby(['term_str', 'pos']).size().to_frame('pos_count')
pos_counts = pos_counts.reset_index()

# Find the most frequent POS tag for each term
pos_max = pos_counts.loc[pos_counts.groupby('term_str')['pos_count'].idxmax()]
pos_max = pos_max.set_index('term_str')['pos'].to_frame().rename(columns={'pos': 'pos_max'})

# Add to VOCAB
VOCAB = VOCAB.join(pos_max, on='term_str')

VOCAB.sample(10)

,term_str,count,stop,porter_stem,pos_max
term_id,,,,,
18812,tests,2,0,test,NNS
21001,whining,1,0,whine,NN
5576,dollop,17,0,dollop,NNP
2772,caucasus,1,0,caucasu,NNP
7082,ferret,4,0,ferret,JJ
5184,dinginess,2,0,dingi,NN
16629,setters,2,0,setter,NNS
2729,casket,2,0,casket,NN
12171,mutilating,1,0,mutil,VBG


In [17]:
# Display final VOCAB structure
VOCAB.head(20)

,term_str,count,stop,porter_stem,pos_max
term_id,,,,,
0,,145853,0,,","
1,1,1,0,1,CD
2,1790,1,0,1790,CD
3,1799,2,0,1799,CD
4,1801,1,0,1801,CD
5,1807,1,0,1807,CD
6,1825,1,0,1825,CD
7,1826,1,0,1826,CD
8,1828,1,0,1828,CD


## Save Tables as CSV

Export all tables to CSV files.

In [18]:
# Drop pos_tuple from TOKEN before saving
TOKEN_save = TOKEN.drop('pos_tuple', axis=1)

# Save all tables
LIBRARY.to_csv('LIBRARY.csv')
DOC.to_csv('DOC.csv')
TOKEN_save.to_csv('TOKEN.csv')
VOCAB.to_csv('VOCAB.csv')

print("All tables saved successfully!")
print(f"  LIBRARY: {len(LIBRARY)} books")
print(f"  DOC: {len(DOC):,} paragraphs")
print(f"  TOKEN: {len(TOKEN):,} tokens")
print(f"  VOCAB: {len(VOCAB):,} unique terms")

All tables saved successfully!
  LIBRARY: 3 books
  DOC: 10,328 paragraphs
  TOKEN: 905,316 tokens
  VOCAB: 21,486 unique terms


## Summary

This notebook successfully created an F3-level digital analytical edition from three George Eliot novels:

1. **LIBRARY Table**: Contains metadata for 3 books
2. **DOC Table**: Contains paragraphs indexed by book_id, chap_num, and para_num
3. **TOKEN Table**: Contains tokens with POS tags indexed by the full OHCO
4. **VOCAB Table**: Contains unique terms with:
   - Stopword flags
   - Porter stems
   - pos_max (most frequent POS tag)

All tables have been saved as CSV files for submission.